## Introduction

This notebook generates copy-paste-ready **README sections** and a compact results table for Scenarios A–C:
- Classical baseline vs QAOA (Aer/local where applicable)
- Braket SV1 runs (where applicable)
- Overlap/Jaccard where available

Outputs:
- `data/results/10a_readme_results_blocks.md` (paste into README)
- `data/results/10a_results_summary_table.csv`


In [1]:
# ============================================================
# Cell 1 — Setup: imports + paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path("data/results")
OUT_MD  = RESULTS_DIR / "10a_readme_results_blocks.md"
OUT_CSV = RESULTS_DIR / "10a_results_summary_table.csv"

# Scenario artifacts (adjust if your filenames differ)
A_COMPARE = RESULTS_DIR / "05c2_compare_best_by_method.csv"  # Scenario A compare
B_COMPARE = RESULTS_DIR / "07c_best_by_method.csv"           # Scenario B compare
C_COMPARE = RESULTS_DIR / "09d_compare_classical_vs_sv1_scenario_C.csv"
C_OVERLAP = RESULTS_DIR / "09d_overlap_classical_vs_sv1_scenario_C.csv"

paths = [A_COMPARE, B_COMPARE, C_COMPARE]
missing = [str(p) for p in paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing:\n- " + "\n- ".join(missing))

print("OK: Found compare artifacts for A, B, C.")


OK: Found compare artifacts for A, B, C.


### What Cell 1 Just Did

- Declared input comparison artifacts for Scenarios A–C (from prior notebooks).
- Declared two outputs: a markdown block file for README + a CSV summary table.
- Fails fast if any required inputs are missing.


In [2]:
# ============================================================
# Cell 2 — Load tables and normalize
# ============================================================

def load_df(p: Path) -> pd.DataFrame:
    df = pd.read_csv(p)
    df["_source_file"] = p.name
    return df

dfa = load_df(A_COMPARE)
dfb = load_df(B_COMPARE)
dfc = load_df(C_COMPARE)

# normalize common columns
def normalize(df: pd.DataFrame, scenario: str) -> pd.DataFrame:
    df = df.copy()
    if "scenario" not in df.columns:
        df["scenario"] = scenario
    # standardize method names
    if "method" not in df.columns:
        # some tables might use "Method"
        for alt in ["Method", "solver", "approach"]:
            if alt in df.columns:
                df["method"] = df[alt]
                break
    # pick best available metrics
    if "energy" not in df.columns:
        for alt in ["qubo_energy_full", "qubo_energy", "objective", "objective_mean", "objective_best"]:
            if alt in df.columns:
                df["energy"] = df[alt]
                break
    if "selected_n" not in df.columns:
        for alt in ["selected_n_mean", "k", "K"]:
            if alt in df.columns:
                df["selected_n"] = df[alt]
                break
    return df

dfa = normalize(dfa, "A")
dfb = normalize(dfb, "B")
dfc = normalize(dfc, "C")

all_df = pd.concat([dfa, dfb, dfc], ignore_index=True)

cols = [c for c in ["scenario", "method", "energy", "selected_n", "_source_file"] if c in all_df.columns]
summary = all_df[cols].copy()

summary.to_csv(OUT_CSV, index=False)
display(summary.head(20))
print("Wrote:", OUT_CSV)


,scenario,method,energy,selected_n,_source_file
0,A,05a_greedy,-1.441677,25,05c2_compare_best_by_method.csv
1,A,05b_qaoa,-1.951298,6,05c2_compare_best_by_method.csv
2,A,05b2_qaoa_aer,12.000000,6,05c2_compare_best_by_method.csv
3,B,greedy,0.000000,9,07c_best_by_method.csv
4,B,greedy,0.000000,9,07c_best_by_method.csv
5,B,greedy,0.000000,9,07c_best_by_method.csv
6,B,greedy,9.000000,9,07c_best_by_method.csv
7,B,greedy,9.000000,9,07c_best_by_method.csv
8,B,greedy,9.000000,9,07c_best_by_method.csv
9,B,greedy,27.000000,9,07c_best_by_method.csv


Wrote: data/results/10a_results_summary_table.csv


### What Cell 2 Just Did

- Loaded the scenario comparison CSVs and normalized them into a single combined table.
- Exported a compact “results summary” CSV suitable for README and slides.


In [3]:
# ============================================================
# Cell 3 — Create paste-ready README blocks (Markdown)
# ============================================================

# Scenario C overlap stats (if available)
jacc = None
if C_OVERLAP.exists():
    overlap = pd.read_csv(C_OVERLAP)
    n_overlap = len(overlap)
    # We can infer jaccard from the 09d notebook output if you also saved stats;
    # otherwise present overlap count.
    jacc = f"{n_overlap} overlapping selections"

def block_for(scenario: str, df: pd.DataFrame) -> str:
    d = df[df["scenario"] == scenario].copy()
    lines = []
    lines.append(f"### Scenario {scenario} — Results")
    lines.append("")
    lines.append("| Method | Energy (or objective proxy) | Selected (n) | Source |")
    lines.append("|---|---:|---:|---|")
    for _, r in d.iterrows():
        m = str(r.get("method", ""))
        e = r.get("energy", np.nan)
        k = r.get("selected_n", np.nan)
        src = str(r.get("_source_file", ""))
        lines.append(f"| {m} | {e:.6g} | {int(k) if pd.notna(k) else ''} | `{src}` |")
    lines.append("")
    if scenario == "C" and jacc:
        lines.append(f"Overlap (Classical vs SV1): {jacc}")
        lines.append("")
    return "\n".join(lines)

md = []
md.append("## Results Summary\n")
md.append(block_for("A", summary))
md.append(block_for("B", summary))
md.append(block_for("C", summary))

OUT_MD.write_text("\n\n".join(md))
print("Wrote:", OUT_MD)
print("\n--- Preview ---\n")
print("\n\n".join(md[:2]))


Wrote: data/results/10a_readme_results_blocks.md

--- Preview ---

## Results Summary


### Scenario A — Results

| Method | Energy (or objective proxy) | Selected (n) | Source |
|---|---:|---:|---|
| 05a_greedy | -1.44168 | 25 | `05c2_compare_best_by_method.csv` |
| 05b_qaoa | -1.9513 | 6 | `05c2_compare_best_by_method.csv` |
| 05b2_qaoa_aer | 12 | 6 | `05c2_compare_best_by_method.csv` |



### What Cell 3 Just Did

- Generated copy-paste-ready Markdown blocks for README:
  - One mini table per scenario (A/B/C).
- Wrote the blocks to `data/results/10a_readme_results_blocks.md`.
- This file is designed to be pasted directly into your `README.md`.


## Summary

Artifacts:
- `data/results/10a_results_summary_table.csv`: a compact cross-scenario summary.
- `data/results/10a_readme_results_blocks.md`: ready-to-paste README result sections.

Next step:
- Paste the generated blocks into `README.md`
- Commit + push README update for a clean showcase finish.
